# Finding Locations

In [3]:
# connect to Enterprise GIS
from arcgis.gis import GIS
import arcgis.geoanalytics

portal_gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)

In [4]:
bigdata_datastore_manager = arcgis.geoanalytics.get_datastores()
bigdata_datastore_manager

<DatastoreManager for https://ndhwks6.esri.com:6443/arcgis/admin>

In [5]:
data_item2 = bigdata_datastore_manager.add_bigdata("all_hurricanes", r"\\NDHWKS6\Users\arcgis\Documents\hurricanes_1848_2010")

Created Big Data file share for all_hurricanes


In [9]:
search_result = portal_gis.content.search("bigDataFileShares_all_hurricanes", 
                                          item_type = "big data file share", 
                                          max_items=40)
search_result

[<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>]

In [10]:
data_item = search_result[0]
data_item

<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>

In [11]:
#displays layers in the item
data_item.layers

[<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_all_hurricanes/BigDataCatalogServer/hurricanes">]

In [12]:
hurricanes = data_item.layers[0] #select first layer 
hurricanes

<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_all_hurricanes/BigDataCatalogServer/hurricanes">

In [13]:
search_result = portal_gis.content.get('8f2fd1d2488f47adbe07a3ddcb05e24e')

In [14]:
table = search_result.tables[0]
table

<Table url:"https://ndhwks6.esri.com/server/rest/services/Hosted/ImportantPlaces/FeatureServer/1">

## Detect Incidents

In [15]:
from arcgis.geoanalytics.find_locations import detect_incidents

This example finds when and where hurricanes are moving where 

In [16]:
##usage example
incidents_detected = detect_incidents(input_layer=hurricanes, 
                                      track_fields='track_type',
                                      start_condition_expression='$feature["Wind"] < 0.2')

In [17]:
incidents_detected

<Item title:"Detect_Incidents_9289SL" type:Feature Layer Collection owner:admin>

## Geocode Locations

In [18]:
from arcgis.geoanalytics.find_locations import geocode_locations

In [19]:
table.query(as_df=True)

,zip,city,street,place,state,objectid
0,91761,Ontario,4511 E Guasti Road,Mark's job,CA,1


In [68]:
from arcgis.geoprocessing import import_toolbox as _import_toolbox
from arcgis.gis import GIS




gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)



print(gis.users.me)



url = gis.properties.helperServices.geoanalytics.url
tbx = _import_toolbox(url, gis=gis)

<User username:admin>


In [71]:
tbx.geocode_locations(input_layer=table, country='CA', output_name='geocoded')

TypeError: geocode_locations() got an unexpected keyword argument 'country'

In [70]:
hasattr(tbx, "geocode_locations")

True

In [21]:
geocoded_locs = geocode_locations(input_layer=table, country='CA', output_name='geocoded')
geocoded_locs

AttributeError: 'Toolbox' object has no attribute 'geocode_locations'

## Find Dwell Locations

In [22]:
from arcgis.geoanalytics.find_locations import find_dwell_locations

In [23]:
dwell_locs = find_dwell_locations(input_layer=hurricanes,
                                  track_fields='track_type',
                                  distance_tolerance=1,
                                  distance_unit='Meters',
                                  time_tolerance='1',
                                  time_unit='Hours', 
                                  output_name='dwell locations')

{"messageCode":"BD_101024","message":"Using geodesic method for geographic coordinate system."}
{"messageCode":"BD_101051","message":"Possible issues were found while reading 'inputLayer'.","params":{"paramName":"inputLayer"}}
{"messageCode":"BD_101052","message":"Some records have either missing or invalid time values."}


In [24]:
dwell_locs

<Item title:"dwell_locations" type:Feature Layer Collection owner:admin>

## Find Similar Locations

In [65]:
from arcgis.geoanalytics.find_locations import find_similar_locations 

In [ ]:
data_item2 = bigdata_datastore_manager.add_bigdata("Chicago_Crimes", r"\\NDHWKS6\Users\arcgis\Documents\ga-store1")

In [60]:
search_result = portal_gis.content.search("bigDataFileShares_Chicago_Crimes", 
                                          item_type = "big data file share", 
                                          max_items=40)
search_result

[<Item title:"bigDataFileShares_Chicago_Crimes_2" type:Big Data File Share owner:admin>,
 <Item title:"bigDataFileShares_Chicago_Crimes" type:Big Data File Share owner:admin>]

In [58]:
homicides = portal_gis.content.get('79a9c31548cb4de2a17b06a9e67095ba')
homicides

<Item title:"Homicides" type:Feature Layer Collection owner:admin>

In [61]:
layers = search_result[0].layers
layers

[<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_Chicago_Crimes_2/BigDataCatalogServer/crime">]

In [63]:
crime_incidents = layers[0]
homicides = homicides.layers[0]

In [66]:
similar_locs = find_similar_locations(input_layer=crime_incidents,
                                      search_layer=homicides,
                                      analysis_fields='Beat',
                                      most_or_least_similar='MostSimilar',
                                      match_method='AttributeValues',
                                      number_of_results=10,
                                      return_tuple=True,
                                      output_name='similar locations')

{"messageCode":"BD_1583","message":"When there are multiple Input Features, matching is based on their averaged Attributes Of Interest."}
{"messageCode":"BD_101051","message":"Possible issues were found while reading 'inputLayer'.","params":{"paramName":"inputLayer"}}
{"messageCode":"BD_101054","message":"Some records have either missing or invalid geometries."}
{"messageCode":"BD_101051","message":"Possible issues were found while reading 'searchLayer'.","params":{"paramName":"searchLayer"}}
{"messageCode":"BD_101054","message":"Some records have either missing or invalid geometries."}


In [67]:
similar_locs.output

<FeatureLayer url:"https://ndhwks6.esri.com/server/rest/services/Hosted/similar_locations/FeatureServer/0">